In [5]:
import numpy as np
import pandas as pd
import os

loss= 'NSE'
test= "PUR"
tests= ['PUB', 'PUR']

# ==========================================================
# LOAD FLOOD ASSIGNMENTS
# ==========================================================

floods_assigned = pd.read_csv(
    "/scratch/s0shar47/floods_assigned_gauges.csv"
)


models = [
    "HBV",
    "HBV_capillary",
    "marrmot_PRMS"
]

# methods= ['USGS', 'Furey', 'Eckhardt', 'LocalHYSEP']
# methods= ['Furey']
methods= [ 'Furey', 'Eckhardt', 'LocalHYSEP']


if test == 'PUB':
    n= 11
elif test == 'PUR':
    n= 8

# ==========================================================
# LOOP MODELS
# ==========================================================
for method in methods:

    for model in models:


        if model == "HBV":
            dyn = "parBETA_parBETAET_"

        elif model == "HBV_capillary":
            dyn = "parBETA_parBETAET_parK0_parUZL_"

        elif model == "marrmot_PRMS":
            dyn = "alpha_cgw_scx_tt_"


        floods_assigned = pd.read_csv("/scratch/s0shar47/floods_assigned_gauges.csv")
        # ======================================================
        # METRIC FUNCTIONS
        # ======================================================

        def calc_nse(obs, sim):

            mask = np.isfinite(obs) & np.isfinite(sim)

            obs = obs[mask]
            sim = sim[mask]

            if len(obs) < 2:
                return np.nan

            den = np.sum(
                (obs - np.mean(obs)) ** 2
            )

            if den == 0:
                return np.nan

            return (
                1 -
                np.sum((obs - sim) ** 2) / den
            )



        def calc_kge(obs, sim):

            mask = np.isfinite(obs) & np.isfinite(sim)

            obs = obs[mask]
            sim = sim[mask]

            if len(obs) < 2:
                return np.nan

            if np.std(obs)==0 or np.mean(obs)==0:
                return np.nan


            r = np.corrcoef(
                obs,
                sim
            )[0,1]


            alpha = (
                np.std(sim)
                /
                np.std(obs)
            )


            beta = (
                np.mean(sim)
                /
                np.mean(obs)
            )


            return (
                1 -
                np.sqrt(
                    (r-1)**2 +
                    (alpha-1)**2 +
                    (beta-1)**2
                )
            )



        def calc_bias(obs, sim):

            mask = np.isfinite(obs) & np.isfinite(sim)

            obs = obs[mask]
            sim = sim[mask]

            if np.sum(obs)==0:
                return np.nan


            return (
                np.sum(sim)-np.sum(obs)
            ) / np.sum(obs)



        def calc_abs_bias(obs, sim):

            b = calc_bias(obs, sim)

            if np.isnan(b):
                return np.nan

            return abs(b)



        # ======================================================
        # LOAD FLOOD EVENTS
        # ======================================================

        flood_events = floods_assigned[
            floods_assigned["MATCH_TYPE"]=="same_county"
        ].copy()


        flood_events["GAGE_ID"] = (
            flood_events["GAGE_ID"]
            .astype(float)
            .astype(int)
        )


        flood_events["Start_Date"] = pd.to_datetime(
            flood_events["Start_Date"]
        )

        flood_events["End_Date"] = pd.to_datetime(
            flood_events["End_Date"]
        )



        # ======================================================
        # GET 531 PUB BASIN GAUGE IDS FROM REGIONAL FORCING
        # ======================================================

        model_gauges = []


        for region in range(1,n):
            if test=='PUB':
                region= str(region)
            else:
                region= f"Region_{region}"

            region_forcing = np.load(
                f"/scratch/data/{test}_DATA/test/{region}/"
                f"daymet_19800101_20141231_forcing_671_baseflow_mmday.npy",
                allow_pickle=True
            )


            model_gauges.extend(
                region_forcing[:,0,14]
                .astype(int)
            )


        model_gauges = set(model_gauges)



        # keep only 531 basin floods

        flood_events = flood_events[
            flood_events["GAGE_ID"].isin(model_gauges)
        ].copy()



        print("--------------------------------")
        print("Model:", model)
        print(
            "Flood events used:",
            len(flood_events)
        )
        print(
            "Unique gauges:",
            flood_events["GAGE_ID"].nunique()
        )



        # ======================================================
        # METRIC TABLE
        # ======================================================

        def metric_table(
            obs_all,
            single_all,
            multi_all,
            events
        ):


            obs_all = np.concatenate(obs_all)

            single_all = np.concatenate(single_all)

            multi_all = np.concatenate(multi_all)


            return {


                "Flood Days":
                    events,


                "Single NSE":
                    calc_nse(
                        obs_all,
                        single_all
                    ),


                "Multi NSE":
                    calc_nse(
                        obs_all,
                        multi_all
                    ),


                "Single KGE":
                    calc_kge(
                        obs_all,
                        single_all
                    ),


                "Multi KGE":
                    calc_kge(
                        obs_all,
                        multi_all
                    ),


                "Single Bias":
                    calc_bias(
                        obs_all,
                        single_all
                    ),


                "Multi Bias":
                    calc_bias(
                        obs_all,
                        multi_all
                    ),


                "Single Abs Bias":
                    calc_abs_bias(
                        obs_all,
                        single_all
                    ),


                "Multi Abs Bias":
                    calc_abs_bias(
                        obs_all,
                        multi_all
                    ),


                "Samples":
                    len(obs_all)

            }


        # Part 1 code
        # metric functions
        # flood filtering
        # metric_table


        # ==============================
        # STORAGE
        # ==============================

        obs_all_events = []
        single_all_events = []
        multi_all_events = []

        total_flood_days = 0


            # ==============================
        # LOOP REGIONS
        # ==============================

        for r in range(1,n):

            # print(f"Processing Region {region}")


            # --------------------------------------------------
            # REGION SPECIFIC FORCING
            # --------------------------------------------------
            if test=='PUR':
                region= f'Region_{r}'
            if test=='PUB':
                region= r
        
            region_forcing = np.load(
                    f"/scratch/data/{test}_DATA/test/{region}/"
                    f"daymet_19800101_20141231_forcing_671_baseflow_mmday.npy",
                    allow_pickle=True
                )

            

            gauge_ids = (
                region_forcing[:,0,14]
                .astype(int)
            )



            # --------------------------------------------------
            # LOAD MODEL OUTPUTS
            # --------------------------------------------------

            # folder = (
            #     f"/home/s0shar47/scratch/"
            #     f"OUTPUTS_Paper1/{test}/"
            #     f"{method}_Multi/"
            #     f"{region}/"
            # )

            folder= f"/scratch/s0shar47//{test}/{method}_Multi/{loss}/{model}/{region}/"

            sfolder= f"/scratch/s0shar47/{test}/Single/{loss}/{model}/{region}/"

            # sfolder = (
            #     f"/home/s0shar47/scratch/"
            #     f"OUTPUTS_Paper1/{test}/"
            #     f"OUTPUTS_{test}_Single/"
            #     f"{region}/"
            # )


            model_folder = (
                f"LSTM_{model}_E50_R365_B100_H256_"
                f"tr1989_1999_n16_111111/"
            )



            obs = np.load(
                os.path.join(
                    folder,
                    model_folder,
                    dyn,
                    "ts1980_2014",
                    "00060_Mean.npy"
                    # 'baseflow.npy'
                )
            )


            sim_multi = np.squeeze(
                np.load(
                    os.path.join(
                        folder,
                        model_folder,
                        dyn,
                        "ts1980_2014",
                        "flow_sim.npy"
                        # "gwflow.npy"
                    )
                )
            )


            sim_single = np.squeeze(
                np.load(
                    os.path.join(
                        sfolder,
                        model_folder,
                        dyn,
                        "ts1980_2014",
                        "flow_sim.npy"
                        # "gwflow.npy"
                    )
                )
            )



            dates = pd.date_range(
                "1981-01-01",
                periods=obs.shape[0],
                freq="D"
            )



            # --------------------------------------------------
            # LOOP BASINS IN THIS REGION
            # --------------------------------------------------

            for basin in range(obs.shape[1]):


                gauge_id = gauge_ids[basin]


                basin_events = flood_events[
                    flood_events["GAGE_ID"] == gauge_id
                ]


                if basin_events.empty:
                    continue



                event_mask = np.zeros(
                    obs.shape[0],
                    dtype=bool
                )



                # ----------------------------------------------
                # CREATE FLOOD WINDOW
                # 3 days before + 3 days after
                # ----------------------------------------------

                for _,event in basin_events.iterrows():

                    start = (
                        event["Start_Date"]
                        -
                        pd.Timedelta(days=2)
                    )

                    end = (
                        event["End_Date"]
                        +
                        pd.Timedelta(days=2)
                    )


                    event_mask |= (
                        (dates >= start)
                        &
                        (dates <= end)
                    )



                # remove missing observations

                event_mask &= np.isfinite(
                    obs[:,basin]
                )


                if not np.any(event_mask):
                    continue



                obs_all_events.append(
                    obs[event_mask,basin]
                )


                single_all_events.append(
                    sim_single[event_mask,basin]
                )


                multi_all_events.append(
                    sim_multi[event_mask,basin]
                )


                total_flood_days += (
                    np.sum(event_mask)
                )



        # ==============================
        # FINAL RESULT (ALL REGIONS)
        # ==============================


        results = metric_table(
            obs_all_events,
            single_all_events,
            multi_all_events,
            total_flood_days
        )



        summary = pd.DataFrame(
            results,
            index=["Flood Events"]
        )


        print(
            "\n=============================="
        )

        print(
            "MODEL:",
            model + " METHOD:",
                        method+  " Test:",
                        test
        )

        print(
            "Total flood samples:",
            total_flood_days
        )

        print(
            "=============================="
        )


        print(f"Loss:",loss)

        display(summary)

--------------------------------
Model: HBV
Flood events used: 2175
Unique gauges: 317

MODEL: HBV METHOD: Furey Test: PUR
Total flood samples: 9479
Loss: NSE


,Flood Days,Single NSE,Multi NSE,Single KGE,Multi KGE,Single Bias,Multi Bias,Single Abs Bias,Multi Abs Bias,Samples
Flood Events,9479,0.525566,0.522921,0.66956,0.642973,-0.106814,-0.124785,0.106814,0.124785,9479


--------------------------------
Model: HBV_capillary
Flood events used: 2175
Unique gauges: 317

MODEL: HBV_capillary METHOD: Furey Test: PUR
Total flood samples: 9479
Loss: NSE


,Flood Days,Single NSE,Multi NSE,Single KGE,Multi KGE,Single Bias,Multi Bias,Single Abs Bias,Multi Abs Bias,Samples
Flood Events,9479,0.525747,0.517845,0.689413,0.653451,-0.102649,-0.127766,0.102649,0.127766,9479


--------------------------------
Model: marrmot_PRMS
Flood events used: 2175
Unique gauges: 317

MODEL: marrmot_PRMS METHOD: Furey Test: PUR
Total flood samples: 9479
Loss: NSE


...

,Flood Days,Single NSE,Multi NSE,Single KGE,Multi KGE,Single Bias,Multi Bias,Single Abs Bias,Multi Abs Bias,Samples
Flood Events,9479,0.525566,0.522212,0.66956,0.638363,-0.106814,-0.135649,0.106814,0.135649,9479


--------------------------------
Model: HBV_capillary
Flood events used: 2175
Unique gauges: 317

MODEL: HBV_capillary METHOD: LocalHYSEP Test: PUR
Total flood samples: 9479
Loss: NSE


,Flood Days,Single NSE,Multi NSE,Single KGE,Multi KGE,Single Bias,Multi Bias,Single Abs Bias,Multi Abs Bias,Samples
Flood Events,9479,0.525747,0.521417,0.689413,0.663755,-0.102649,-0.121567,0.102649,0.121567,9479


--------------------------------
Model: marrmot_PRMS
Flood events used: 2175
Unique gauges: 317

MODEL: marrmot_PRMS METHOD: LocalHYSEP Test: PUR
Total flood samples: 9479
Loss: NSE


,Flood Days,Single NSE,Multi NSE,Single KGE,Multi KGE,Single Bias,Multi Bias,Single Abs Bias,Multi Abs Bias,Samples
Flood Events,9479,0.530527,0.532504,0.654833,0.63516,-0.126173,-0.130352,0.126173,0.130352,9479


In [6]:
import numpy as np
import pandas as pd
import os

loss= 'RMSE'
test= "PUR"
tests= ['PUB', 'PUR']

# ==========================================================
# LOAD FLOOD ASSIGNMENTS
# ==========================================================

floods_assigned = pd.read_csv(
    "/scratch/s0shar47/floods_assigned_gauges.csv"
)


models = [
    "HBV",
    "HBV_capillary",
    "marrmot_PRMS"
]

# methods= ['USGS', 'Furey', 'Eckhardt', 'LocalHYSEP']
# methods= ['Furey']
methods= [ 'Furey', 'Eckhardt', 'LocalHYSEP']


if test == 'PUB':
    n= 11
elif test == 'PUR':
    n= 8

# ==========================================================
# LOOP MODELS
# ==========================================================
for method in methods:

    for model in models:


        if model == "HBV":
            dyn = "parBETA_parBETAET_"

        elif model == "HBV_capillary":
            dyn = "parBETA_parBETAET_parK0_parUZL_"

        elif model == "marrmot_PRMS":
            dyn = "alpha_cgw_scx_tt_"


        floods_assigned = pd.read_csv("/scratch/s0shar47/floods_assigned_gauges.csv")
        # ======================================================
        # METRIC FUNCTIONS
        # ======================================================

        def calc_nse(obs, sim):

            mask = np.isfinite(obs) & np.isfinite(sim)

            obs = obs[mask]
            sim = sim[mask]

            if len(obs) < 2:
                return np.nan

            den = np.sum(
                (obs - np.mean(obs)) ** 2
            )

            if den == 0:
                return np.nan

            return (
                1 -
                np.sum((obs - sim) ** 2) / den
            )



        def calc_kge(obs, sim):

            mask = np.isfinite(obs) & np.isfinite(sim)

            obs = obs[mask]
            sim = sim[mask]

            if len(obs) < 2:
                return np.nan

            if np.std(obs)==0 or np.mean(obs)==0:
                return np.nan


            r = np.corrcoef(
                obs,
                sim
            )[0,1]


            alpha = (
                np.std(sim)
                /
                np.std(obs)
            )


            beta = (
                np.mean(sim)
                /
                np.mean(obs)
            )


            return (
                1 -
                np.sqrt(
                    (r-1)**2 +
                    (alpha-1)**2 +
                    (beta-1)**2
                )
            )



        def calc_bias(obs, sim):

            mask = np.isfinite(obs) & np.isfinite(sim)

            obs = obs[mask]
            sim = sim[mask]

            if np.sum(obs)==0:
                return np.nan


            return (
                np.sum(sim)-np.sum(obs)
            ) / np.sum(obs)



        def calc_abs_bias(obs, sim):

            b = calc_bias(obs, sim)

            if np.isnan(b):
                return np.nan

            return abs(b)



        # ======================================================
        # LOAD FLOOD EVENTS
        # ======================================================

        flood_events = floods_assigned[
            floods_assigned["MATCH_TYPE"]=="same_county"
        ].copy()


        flood_events["GAGE_ID"] = (
            flood_events["GAGE_ID"]
            .astype(float)
            .astype(int)
        )


        flood_events["Start_Date"] = pd.to_datetime(
            flood_events["Start_Date"]
        )

        flood_events["End_Date"] = pd.to_datetime(
            flood_events["End_Date"]
        )



        # ======================================================
        # GET 531 PUB BASIN GAUGE IDS FROM REGIONAL FORCING
        # ======================================================

        model_gauges = []


        for region in range(1,n):
            if test=='PUB':
                region= str(region)
            else:
                region= f"Region_{region}"

            region_forcing = np.load(
                f"/scratch/data/{test}_DATA/test/{region}/"
                f"daymet_19800101_20141231_forcing_671_baseflow_mmday.npy",
                allow_pickle=True
            )


            model_gauges.extend(
                region_forcing[:,0,14]
                .astype(int)
            )


        model_gauges = set(model_gauges)



        # keep only 531 basin floods

        flood_events = flood_events[
            flood_events["GAGE_ID"].isin(model_gauges)
        ].copy()



        print("--------------------------------")
        print("Model:", model)
        print(
            "Flood events used:",
            len(flood_events)
        )
        print(
            "Unique gauges:",
            flood_events["GAGE_ID"].nunique()
        )



        # ======================================================
        # METRIC TABLE
        # ======================================================

        def metric_table(
            obs_all,
            single_all,
            multi_all,
            events
        ):


            obs_all = np.concatenate(obs_all)

            single_all = np.concatenate(single_all)

            multi_all = np.concatenate(multi_all)


            return {


                "Flood Days":
                    events,


                "Single NSE":
                    calc_nse(
                        obs_all,
                        single_all
                    ),


                "Multi NSE":
                    calc_nse(
                        obs_all,
                        multi_all
                    ),


                "Single KGE":
                    calc_kge(
                        obs_all,
                        single_all
                    ),


                "Multi KGE":
                    calc_kge(
                        obs_all,
                        multi_all
                    ),


                "Single Bias":
                    calc_bias(
                        obs_all,
                        single_all
                    ),


                "Multi Bias":
                    calc_bias(
                        obs_all,
                        multi_all
                    ),


                "Single Abs Bias":
                    calc_abs_bias(
                        obs_all,
                        single_all
                    ),


                "Multi Abs Bias":
                    calc_abs_bias(
                        obs_all,
                        multi_all
                    ),


                "Samples":
                    len(obs_all)

            }


        # Part 1 code
        # metric functions
        # flood filtering
        # metric_table


        # ==============================
        # STORAGE
        # ==============================

        obs_all_events = []
        single_all_events = []
        multi_all_events = []

        total_flood_days = 0


            # ==============================
        # LOOP REGIONS
        # ==============================

        for r in range(1,n):

            # print(f"Processing Region {region}")


            # --------------------------------------------------
            # REGION SPECIFIC FORCING
            # --------------------------------------------------
            if test=='PUR':
                region= f'Region_{r}'
            if test=='PUB':
                region= r
        
            region_forcing = np.load(
                    f"/scratch/data/{test}_DATA/test/{region}/"
                    f"daymet_19800101_20141231_forcing_671_baseflow_mmday.npy",
                    allow_pickle=True
                )

            

            gauge_ids = (
                region_forcing[:,0,14]
                .astype(int)
            )



            # --------------------------------------------------
            # LOAD MODEL OUTPUTS
            # --------------------------------------------------

            # folder = (
            #     f"/home/s0shar47/scratch/"
            #     f"OUTPUTS_Paper1/{test}/"
            #     f"{method}_Multi/"
            #     f"{region}/"
            # )

            folder= f"/scratch/s0shar47//{test}/{method}_Multi/{loss}/{model}/{region}/"

            sfolder= f"/scratch/s0shar47/{test}/Single/{loss}/{model}/{region}/"

            # sfolder = (
            #     f"/home/s0shar47/scratch/"
            #     f"OUTPUTS_Paper1/{test}/"
            #     f"OUTPUTS_{test}_Single/"
            #     f"{region}/"
            # )


            model_folder = (
                f"LSTM_{model}_E50_R365_B100_H256_"
                f"tr1989_1999_n16_111111/"
            )



            obs = np.load(
                os.path.join(
                    folder,
                    model_folder,
                    dyn,
                    "ts1980_2014",
                    "00060_Mean.npy"
                    # 'baseflow.npy'
                )
            )


            sim_multi = np.squeeze(
                np.load(
                    os.path.join(
                        folder,
                        model_folder,
                        dyn,
                        "ts1980_2014",
                        "flow_sim.npy"
                        # "gwflow.npy"
                    )
                )
            )


            sim_single = np.squeeze(
                np.load(
                    os.path.join(
                        sfolder,
                        model_folder,
                        dyn,
                        "ts1980_2014",
                        "flow_sim.npy"
                        # "gwflow.npy"
                    )
                )
            )



            dates = pd.date_range(
                "1981-01-01",
                periods=obs.shape[0],
                freq="D"
            )



            # --------------------------------------------------
            # LOOP BASINS IN THIS REGION
            # --------------------------------------------------

            for basin in range(obs.shape[1]):


                gauge_id = gauge_ids[basin]


                basin_events = flood_events[
                    flood_events["GAGE_ID"] == gauge_id
                ]


                if basin_events.empty:
                    continue



                event_mask = np.zeros(
                    obs.shape[0],
                    dtype=bool
                )



                # ----------------------------------------------
                # CREATE FLOOD WINDOW
                # 3 days before + 3 days after
                # ----------------------------------------------

                for _,event in basin_events.iterrows():

                    start = (
                        event["Start_Date"]
                        -
                        pd.Timedelta(days=2)
                    )

                    end = (
                        event["End_Date"]
                        +
                        pd.Timedelta(days=2)
                    )


                    event_mask |= (
                        (dates >= start)
                        &
                        (dates <= end)
                    )



                # remove missing observations

                event_mask &= np.isfinite(
                    obs[:,basin]
                )


                if not np.any(event_mask):
                    continue



                obs_all_events.append(
                    obs[event_mask,basin]
                )


                single_all_events.append(
                    sim_single[event_mask,basin]
                )


                multi_all_events.append(
                    sim_multi[event_mask,basin]
                )


                total_flood_days += (
                    np.sum(event_mask)
                )



        # ==============================
        # FINAL RESULT (ALL REGIONS)
        # ==============================


        results = metric_table(
            obs_all_events,
            single_all_events,
            multi_all_events,
            total_flood_days
        )



        summary = pd.DataFrame(
            results,
            index=["Flood Events"]
        )


        print(
            "\n=============================="
        )

        print(
            "MODEL:",
            model + " METHOD:",
                        method+  " Test:",
                        test
        )

        print(
            "Total flood samples:",
            total_flood_days
        )

        print(
            "=============================="
        )


        print(f"Loss:",loss)

        display(summary)

--------------------------------
Model: HBV
Flood events used: 2175
Unique gauges: 317

MODEL: HBV METHOD: Furey Test: PUR
Total flood samples: 9479
Loss: RMSE


,Flood Days,Single NSE,Multi NSE,Single KGE,Multi KGE,Single Bias,Multi Bias,Single Abs Bias,Multi Abs Bias,Samples
Flood Events,9479,0.53423,0.524249,0.66369,0.648903,-0.115242,-0.12989,0.115242,0.12989,9479


--------------------------------
Model: HBV_capillary
Flood events used: 2175
Unique gauges: 317

MODEL: HBV_capillary METHOD: Furey Test: PUR
Total flood samples: 9479
Loss: RMSE


,Flood Days,Single NSE,Multi NSE,Single KGE,Multi KGE,Single Bias,Multi Bias,Single Abs Bias,Multi Abs Bias,Samples
Flood Events,9479,0.49675,0.5148,0.693535,0.683985,-0.094501,-0.109309,0.094501,0.109309,9479


--------------------------------
Model: marrmot_PRMS
Flood events used: 2175
Unique gauges: 317

MODEL: marrmot_PRMS METHOD: Furey Test: PUR
Total flood samples: 9479
Loss: RMSE


...

,Flood Days,Single NSE,Multi NSE,Single KGE,Multi KGE,Single Bias,Multi Bias,Single Abs Bias,Multi Abs Bias,Samples
Flood Events,9479,0.53423,0.54839,0.66369,0.672431,-0.115242,-0.116607,0.115242,0.116607,9479


--------------------------------
Model: HBV_capillary
Flood events used: 2175
Unique gauges: 317

MODEL: HBV_capillary METHOD: LocalHYSEP Test: PUR
Total flood samples: 9479
Loss: RMSE


,Flood Days,Single NSE,Multi NSE,Single KGE,Multi KGE,Single Bias,Multi Bias,Single Abs Bias,Multi Abs Bias,Samples
Flood Events,9479,0.49675,0.534941,0.693535,0.689188,-0.094501,-0.112023,0.094501,0.112023,9479


--------------------------------
Model: marrmot_PRMS
Flood events used: 2175
Unique gauges: 317

MODEL: marrmot_PRMS METHOD: LocalHYSEP Test: PUR
Total flood samples: 9479
Loss: RMSE


,Flood Days,Single NSE,Multi NSE,Single KGE,Multi KGE,Single Bias,Multi Bias,Single Abs Bias,Multi Abs Bias,Samples
Flood Events,9479,0.52212,0.53897,0.681382,0.683615,-0.110741,-0.111719,0.110741,0.111719,9479


In [7]:
obs_all_events

[array([ 5.5875325,  4.691821 ,  4.1373334,  5.5875325, 25.634405 ,
        27.937662 ], dtype=float32),
 array([ 4.040239  ,  3.403981  ,  2.385968  ,  2.134646  , 23.414299  ,
         1.0339195 ,  1.0052878 ,  0.92575556, 21.314648  , 27.931732  ,
         3.372168  ,  2.7868106 ,  2.5895705 ,  2.987232  , 16.606337  ,
        15.906453  ,  4.8673744 , 15.492886  ,  9.41662   ,  8.716736  ,
        13.90224   ,  9.225742  ], dtype=float32),
 array([ 1.4290282 ,  1.5797354 ,  1.4478666 ,  1.227188  , 17.950315  ,
        20.372396  ,  4.1982746 ,  3.2294421 ,  2.7988498 ,  3.06797   ,
        28.52674   , 18.757677  , 24.893616  , 13.294537  ,  9.795975  ,
         7.4815407 ,  0.37945944,  0.320253  ,  0.9311558 , 31.48706   ,
        22.256239  ,  7.885221  ,  7.9928694 ,  5.812996  ,  5.3285794 ,
        22.498447  , 22.390799  ,  2.6615984 ,  2.4516847 ,  2.3413455 ,
         2.5700977 , 10.307302  , 30.94882   ], dtype=float32),
 array([ 4.544853 ,  6.6292386,  5.9344435,  3.588